In [ ]:
import math
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.gridspec as gridspec
import pandas as pd

from os.path import join
import os
from functools import partial
import pathlib
from pathlib import Path
import shutil

from joblib import Parallel, delayed
import joblib

import gc
import subprocess

from credit.pbs import get_num_cpus

In [ ]:
#parameters
forecast_dir = "/glade/derecho/scratch/dkimpara/goes_10km_train/wx_inverted/forecasts/2022-07-01T00:25:06"
c13_only = 0
num_cpus = 2
test = 0
config = None
delay = 50
dpi = 300
max_steps = 12

In [ ]:
if c13_only:
    channels = [13]
else:
    channels = [7, 8, 9, 10, 13]

In [ ]:
def plot_ensemble_forecast(ds, true_ds, init_ds, forecast_dir, channels, img_idx=-1):
    true_ds["t"] = ds.t
    
    for channel in channels:
        # Extract true and squeeze
        true = true_ds.sel(channel=channel).BT_or_R.squeeze()
        
        # Extract prediction (Keep ensemble dim, squeeze others)
        preds = ds.sel(channel=channel).BT_or_R.squeeze()
        
        if "ensemble_member_label" in preds.dims:
            num_members = preds.ensemble_member_label.size
            member_labels = preds.ensemble_member_label.values
        else:
            num_members = 1
            member_labels = [0]
            
        total_plots = num_members + 1 # +1 for Truth
        cols = 6 # Increased to 7 columns
        rows = math.ceil(total_plots / cols)
        
        # Create dynamic grid
        fig, axes = plt.subplots(
            rows, cols, 
            figsize=(18, 3.0 * rows),
            subplot_kw={'projection': ccrs.PlateCarree()}
        )
        axes = np.atleast_1d(axes).flatten()
        
        if channel > 4:
            cmap, vmin, vmax = "Spectral_r", 200, 300
        else:
            cmap, vmin, vmax = "Spectral_r", 0.0, 0.2

        # 1. Plot Truth in first subplot
        ax = axes[0]
        im = true.plot(ax=ax, transform=ccrs.PlateCarree(), cmap=cmap, vmax=vmax, vmin=vmin, add_colorbar=False)
        ax.set_title("Observations (Truth)")

        # 2. Plot Ensemble Members
        for i in range(num_members):
            ax = axes[i + 1]
            if "ensemble_member_label" in preds.dims:
                member_data = preds.isel(ensemble_member_label=i)
                title = f"Member {member_labels[i]}"
            else:
                member_data = preds
                title = "Prediction"
                
            member_data.plot(ax=ax, transform=ccrs.PlateCarree(), cmap=cmap, vmax=vmax, vmin=vmin, add_colorbar=False)
            ax.set_title(title)

        # 3. Formatting
        for i in range(total_plots):
            ax = axes[i]
            ax.add_feature(cfeature.COASTLINE)
            
            ax.set_xticks(list(range(-120,-29,30)))
            ax.set_yticks(list(range(-50,51,25)))
            
            col_idx = i % cols
            
            # Left edge: Keep ticks, tick numbers, and add 'Latitude' title
            if col_idx == 0:
                ax.set_ylabel("Latitude", fontsize=10)
            else:
                # Reverted hiding of tick marks, but keep hiding tick numbers to prevent overlap
                ax.set_ylabel("")
                ax.tick_params(labelleft=False)
                
            # Bottom edge: Keep ticks, tick numbers, and add 'Longitude' title
            if i + cols >= total_plots:
                ax.set_xlabel("Longitude", fontsize=10)
            else:
                # Reverted hiding of tick marks, but keep hiding tick numbers to prevent overlap
                ax.set_xlabel("")
                ax.tick_params(labelbottom=False) 

        # Hide unused axes if total_plots doesn't perfectly divide by 7
        for i in range(total_plots, len(axes)):
            axes[i].axis('off')
            
        # 4. Shared Colorbar
        # Adjusted y-position dynamically based on rows
        cbar_y = 0.06 / rows 
        cbar_ax = fig.add_axes([0.325, cbar_y, 0.35, 0.02])
        fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
        
        # 5. Title & Save
        time_val = preds.t.values
        if hasattr(time_val, "item"): 
            time_val = time_val.item()
        time_str_title = pd.Timestamp(time_val).strftime('%Y-%m-%dT%H:%M:%S')
        
        # Doubled bottom spacing (0.16/rows instead of 0.08/rows) to push subplots away from the colorbar
        fig.subplots_adjust(top=0.90, bottom=cbar_y + (0.22 / rows), hspace=0.15, wspace=0.1)
        
        fig.suptitle(f"Channel {channel}, forecast step {img_idx:02}\n{time_str_title}", y=0.99, fontsize=14)

        figname = f"C{channel:02}_step{img_idx:02}.png"
        path1 = pathlib.Path(forecast_dir)
        time_str = path1.parent.name if not path1.is_dir() else path1.name
        forecast_parent_dir = path1.parent
        
        # Save to ensemble_gif
        save_dir = join(forecast_parent_dir, "ensemble_gif", f"gifs_{time_str}", f"C{channel:02}")
        os.makedirs(save_dir, exist_ok=True)
        
        plt.savefig(join(save_dir, figname), format="png", dpi=dpi, bbox_inches="tight")
        plt.close(fig)

In [ ]:
num_cpus = max(1, get_num_cpus() - 1)
print(f"using {num_cpus} cpus")
print(f"processing {forecast_dir}")

files = sorted([f for f in os.listdir(forecast_dir) if os.path.isfile(join(forecast_dir, f))])[:max_steps]

if test:
    print("WARNING: test mode!")
    files = files[:2]
    num_cpus = 2
    
def make_gif(forecast_dir, channels, file_and_index):
    k, file = file_and_index
    img_idx = k + 1
    
    zarr_ds = xr.open_dataset("/glade/derecho/scratch/dkimpara/goes-cloud-dataset/goes_10km.zarr", consolidated=False).drop_duplicates(
        dim="t"
    ).sortby("t").transpose("t", "channel", "latitude", "longitude")
    
    path1 = pathlib.Path(forecast_dir)
    time_str = path1.parent.name if not path1.is_dir() else path1.name
    init_time = pd.Timestamp(time_str)
    init_ds = zarr_ds.sel(t=[init_time], method="nearest")

    if img_idx == 0: 
        # Peek at the first actual forecast to get ensemble labels
        dummy_ds = xr.open_dataset(join(forecast_dir, files[0]))
        if "ensemble_member_label" in dummy_ds.dims:
            # Broadcast the initial state N times to match grid layout
            init_ds_ens = init_ds.expand_dims(ensemble_member_label=dummy_ds.ensemble_member_label)
        else:
            init_ds_ens = init_ds
            
        plot_ensemble_forecast(init_ds_ens, init_ds, init_ds, forecast_dir, channels, img_idx)
        gc.collect()
        return

    file = join(forecast_dir, file)
    ds = xr.open_dataset(file) 
    true_ds = zarr_ds.sel(t=ds.t, method="nearest")

    plot_ensemble_forecast(ds, true_ds, init_ds, forecast_dir, channels, img_idx)
    gc.collect()

f = partial(make_gif, forecast_dir, channels)
enum_files = [(-1, None)] + list(enumerate(files))
print(files)

result = Parallel(n_jobs = num_cpus)(delayed(f)(file_and_index) for file_and_index in enum_files)

using 1 cpus
processing /glade/derecho/scratch/dkimpara/goes_10km_train/wx_inverted/forecasts/2022-07-01T00:25:06


: 

In [ ]:
path1 = pathlib.Path(forecast_dir)
time_str = path1.name

# Pointing to the new directory
gif_dir = join(path1.parent, "ensemble_gif", f"gifs_{time_str}")

for channel in channels:
    res = subprocess.run(
                f"magick -delay {delay} -loop 0 {join(gif_dir, f'C{channel:02}/*.png')} {join(gif_dir, f'C{channel:02}.gif')}",
                shell=True,
                capture_output=True,
                encoding="utf-8",
            )
    print(f"created ensemble gif for C{channel:02}")
    shutil.rmtree(join(gif_dir, f'C{channel:02}'))